# Bake-off: overlap vs embeddings

I have to decide how YumOver finds recipes starting from what you have in the fridge. I try two
approaches and compare them:
1. **Overlap**: I look at which ingredients of the recipe I have in the pantry, and score by how
   many are missing.
2. **Embeddings**: I turn recipes and pantry into embeddings and look for the closest ones.

The test queries are 17 and live in `eval-queries.json`.
Each one is a pantry, with the recipes I expect and the ones I would be happy with anyway.
Two pantries have no answer: there, the right thing to do is not to answer.

I made some choices before running anything, so I would not be tempted to adjust them after
seeing the numbers:

| Choice | What I decided |
|---|---|
| How I score | Jaccard: len(matched) / len(needed ∪ have) |
| Staples | Salt, pepper, oil, water: everybody has them |
| Derivates | If you have eggs you also have yolks. Not the other way round |
| When not to answer | Below 2 ingredients in common, overlap stays silent |
| Recipes with the same score | First the one that leaves me less to buy |
| What I feed the model | The list of ingredients. I try three other forms too |

In [1]:
import json
import statistics
from pathlib import Path
from time import perf_counter

import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RECIPES_PATH = ROOT / "data" / "recipes-seed.json"
INGREDIENTS_PATH = ROOT / "data" / "ingredients.json"
QUERIES_PATH = ROOT / "notebooks" / "eval-queries.json"

K = 3
MIN_MATCH = 2

print(ROOT)

/home/alin/Progetti/Project-Food-Coach


In [2]:
recipes = json.loads(RECIPES_PATH.read_text(encoding="utf-8"))
ingredients = json.loads(INGREDIENTS_PATH.read_text(encoding="utf-8"))["ingredients"]
queries = json.loads(QUERIES_PATH.read_text(encoding="utf-8"))

print(f"{len(queries)} queries | {len(recipes)} recipes | {len(ingredients)} ingredients")
assert (len(queries), len(recipes), len(ingredients)) == (17, 30, 80)

17 queries | 30 recipes | 80 ingredients


In [3]:
staples = {i["key"] for i in ingredients if i["is_staple"]}

# base -> {derivati}. Solo in questo verso: le uova danno i tuorli, non viceversa.
derived_of = {}
for i in ingredients:
    if i["derives_from"]:
        derived_of.setdefault(i["derives_from"], set()).add(i["key"])

display = {i["key"]: i["display"] for i in ingredients}
category = {i["key"]: i["category"] for i in ingredients}
titles = {r["id"]: r["title"] for r in recipes}

# Gli staple escono qui: e' il denominatore del punteggio.
# La Focaccia elenca `sale` due volte, e nel set il duplicato collassa da solo.
recipe_keys = {r["id"]: {ing["key"] for ing in r["ingredients"]} - staples for r in recipes}

print("staples:", sorted(staples))
print("derivations:", {k: sorted(v) for k, v in derived_of.items()})
print("Carbonara (#3) without staples:", sorted(recipe_keys[3]))

staples: ['acqua', 'olio-di-oliva', 'pepe', 'sale']
derivations: {'uova': ['tuorli'], 'limone': ['scorza-di-limone', 'succo-di-limone']}
Carbonara (#3) without staples: ['guanciale', 'pecorino', 'spaghetti', 'tuorli']


## Salt, pepper, oil, and eggs

Salt, pepper, oil and water are in almost every recipe, so if I counted them every recipe would
get free points and the scores would all end up close together. I drop them, and I drop them from
the recipe as well as from the pantry: if I dropped them on one side only, no pantry would ever
cover a recipe 100%.

Then there are ingredients you get out of other ones. If you have eggs you also have yolks, and
Carbonara wants yolks. It does not work backwards: from a yolk you don't get back to an egg.

In [4]:
def written(pantry):
    """Non-staple keys the user actually put in the pantry."""
    return {k for k in pantry if k not in staples}


def reach(pantry):
    """Everything the pantry can supply, including what it turns into."""
    have = written(pantry)
    out = set(have)
    for key in have:
        out |= derived_of.get(key, set())
    return out


assert reach(["uova"]) == {"uova", "tuorli"}
assert reach(["tuorli"]) == {"tuorli"}          # non si risale
assert reach(["sale", "acqua"]) == set()
print("reach ok")

reach ok


## How I score

Three ways of scoring the results with a scoring function

| Formula | How it is computed |
|---|---|
| `count` | len(matched) |
| `coverage` |len(matched) / len(needed)|
| `jaccard` | len(matched) / len(needed ∪ have) |

I use the Jaccard index and try the other two as well.

If I have less than 2 ingredients in common I return nothing.

If two recipes get the same score I put first the one that leaves me less to buy.

In [5]:
def score_count(matched, needed, have):
    return float(len(matched))


def score_coverage(matched, needed, have):
    return len(matched) / len(needed) if needed else 0.0


def score_jaccard(matched, needed, have):
    union = needed | have
    return len(matched) / len(union) if union else 0.0


SCORERS = {"jaccard": score_jaccard, "coverage": score_coverage, "count": score_count}


def retrieve_overlap(pantry, scorer="jaccard", k=K, min_match=MIN_MATCH):
    """Return up to k (recipe_id, score) pairs. Empty when nothing clears the threshold."""
    have = written(pantry)
    available = reach(pantry)
    rows = []
    for recipe_id, needed in recipe_keys.items():
        matched = needed & available
        if len(matched) < min_match:
            continue
        missing = len(needed - available)
        rows.append((recipe_id, SCORERS[scorer](matched, needed, have), missing))
    # punteggio alto, poi meno ingredienti da comprare, poi id: sempre lo stesso ordine
    rows.sort(key=lambda row: (-row[1], row[2], row[0]))
    return [(recipe_id, score) for recipe_id, score, _ in rows[:k]]

In [6]:
# Una dispensa di soli staple non produce niente.
assert retrieve_overlap(["sale", "olio-di-oliva", "pepe", "acqua"]) == []

# q17 copre #5 al completo: se non fa 1.0, uno staple e' rimasto nel denominatore.
assert retrieve_overlap(["patate", "rosmarino", "timo", "aglio"], scorer="coverage")[0] == (5, 1.0)

# q5 ha le uova intere, la Carbonara vuole i tuorli.
assert retrieve_overlap(["spaghetti", "uova", "guanciale", "pecorino"])[0][0] == 3

for query in queries[:3]:
    print(query["id"])
    for recipe_id, score in retrieve_overlap(query["pantry"]):
        print(f"   {score:.2f}  #{recipe_id} {titles[recipe_id]}")

q1-classic
   1.00  #3 Pasta alla Carbonara
   0.50  #18 Pasta all'Amatriciana
q2-tiramisu
   0.50  #1 Tiramisù
q3-strange-combination
   0.57  #29 Pasta alle Vongole
   0.25  #7 Pasta al pomodoro
   0.20  #9 Filetti di merluzzo con pomodorini, olive e capperi


## What I feed the model

The model wants a string, and a recipe is an object with 10 fields inside. So I have to choose
what to write to it. I try four forms:

- `A`, ingredients only: `guanciale, pecorino, spaghetti, tuorli`
- `A-key`, the same ones but with the database naming: `riso-da-risotto` instead of `riso da risotto`
- `B`, title and ingredients. But the pantry has no title, so I am comparing two things built in different ways
- `C`, ingredients and category, like `guanciale (meat)`

Recipes and pantry go through the same function, `to_text`.

To the pantry I add the derivates (eggs become yolks too) because I add them to the overlap as
well. Otherwise I give one strategy information I deny to the other, and then I no longer know
what I am comparing.

In [7]:
def to_text(keys, title=None, with_category=False, naming="display"):
    """The only path from a set of keys to a string. Used for recipes AND for the query."""
    parts = []
    for key in sorted(keys):          # un set non ha ordine: senza sorted i vettori cambiano
        name = display.get(key, key) if naming == "display" else key
        if with_category:
            name = f"{name} ({category.get(key, '?')})"
        parts.append(name)
    body = ", ".join(parts)
    return f"{title}: {body}" if title else body


VARIANTS = {
    "A":     dict(with_category=False, use_title=False, naming="display"),
    "A-key": dict(with_category=False, use_title=False, naming="key"),
    "B":     dict(with_category=False, use_title=True,  naming="display"),
    "C":     dict(with_category=True,  use_title=False, naming="display"),
}

print("A recipe :", to_text(recipe_keys[3]))
print("A query  :", to_text(reach(["spaghetti", "uova", "guanciale", "pecorino"])))
print("A-key    :", to_text(recipe_keys[3], naming="key"))
print("B recipe :", to_text(recipe_keys[3], title=titles[3]))
print("C recipe :", to_text(recipe_keys[3], with_category=True))

A recipe : guanciale, pecorino, spaghetti, tuorli
A query  : guanciale, pecorino, spaghetti, tuorli, uova
A-key    : guanciale, pecorino, spaghetti, tuorli
B recipe : Pasta alla Carbonara: guanciale, pecorino, spaghetti, tuorli
C recipe : guanciale (meat), pecorino (dairy), spaghetti (cereals-grains), tuorli (eggs)


In [8]:
import torch
from sentence_transformers import SentenceTransformer

# Senza questa riga l'encoding di una frase corta passa da ~70 ms a ~2800: il coordinamento
# fra thread costa piu' del lavoro. La colonna di latenza sarebbe sbagliata di 40 volte.
torch.set_num_threads(1)

t0 = perf_counter()
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
load_ms = (perf_counter() - t0) * 1000

recipe_ids = [r["id"] for r in recipes]
vectors = {}
encode_ms = {}
for name, cfg in VARIANTS.items():
    texts = [
        to_text(recipe_keys[recipe_id],
                title=titles[recipe_id] if cfg["use_title"] else None,
                with_category=cfg["with_category"],
                naming=cfg["naming"])
        for recipe_id in recipe_ids
    ]
    t0 = perf_counter()
    # normalize: a lunghezza 1 il coseno e' il solo prodotto scalare
    vectors[name] = model.encode(texts, normalize_embeddings=True)
    encode_ms[name] = (perf_counter() - t0) * 1000

print(f"model load: {load_ms:.0f} ms")
for name in VARIANTS:
    print(f"variant {name}: {vectors[name].shape}  encoding 30 recipes {encode_ms[name]:.0f} ms")
assert vectors["A"].shape == (30, 384)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model load: 17799 ms
variant A: (30, 384)  encoding 30 recipes 1400 ms
variant A-key: (30, 384)  encoding 30 recipes 1020 ms
variant B: (30, 384)  encoding 30 recipes 1002 ms
variant C: (30, 384)  encoding 30 recipes 2018 ms


## Embedding the recipes

The 30 recipes become vectors once, at startup. When the user asks for recipes, the only thing to
compute on the spot is the vector of their pantry.

The two strategies pay at two different moments. The embedding pays everything at startup and
then almost nothing; the overlap pays nothing at startup and works on every request. With 5000
recipes the vectors would have to be saved to disk instead of recomputed at every restart, and
that is why projects like this one use a vector database.

In [9]:
def retrieve_embed(pantry, variant="A", k=K, threshold=None):
    """Same signature as retrieve_overlap. No threshold: it always answers."""
    query_text = to_text(reach(pantry),
                         with_category=VARIANTS[variant]["with_category"],
                         naming=VARIANTS[variant]["naming"])
    query_vector = model.encode(query_text, normalize_embeddings=True)
    sims = vectors[variant] @ query_vector      # (30, 384) x (384,) = 30 cosini
    order = np.argsort(-sims, kind="stable")    # stable: a parita' vince l'id piu' basso
    out = []
    for idx in order[:k]:
        score = float(sims[idx])
        if threshold is not None and score < threshold:
            break
        out.append((recipe_ids[idx], score))
    return out


for query in queries[:3]:
    print(query["id"])
    for recipe_id, score in retrieve_embed(query["pantry"]):
        print(f"   {score:.3f}  #{recipe_id} {titles[recipe_id]}")

q1-classic
   1.000  #3 Pasta alla Carbonara
   0.884  #18 Pasta all'Amatriciana
   0.882  #29 Pasta alle Vongole
q2-tiramisu
   0.820  #1 Tiramisù
   0.709  #24 Saltimbocca alla Romana
   0.679  #25 Focaccia Genovese
q3-strange-combination
   0.941  #3 Pasta alla Carbonara
   0.899  #7 Pasta al pomodoro
   0.875  #18 Pasta all'Amatriciana


## How I score the strategies

Every strategy returns 3 recipes. To tell whether they are any good I look at four things.

**recall@3**: of the recipes I expected, how many showed up in the top 3? A recipe you could have
cooked and I did not show you is food that ends up in the bin, which is the problem the app is
supposed to solve.

**precision@3**: of the 3 I showed, how many were good? It is needed because otherwise the best
strategy would be "show all 30", which has perfect recall and is useless.

**MRR**: the position of the first good recipe. First place is worth 1, second 0.5, third 0.33.
The user reads from the top, so position counts.

**silence**: on the two pantries with no answer, how many times it returned 'empty' instead of
making something up.

In [10]:
def judge(query, results):
    """Metrics for ONE query. results = [(recipe_id, score)]."""
    expected = set(query["expected"])
    good = expected | set(query["acceptable"])
    returned = [recipe_id for recipe_id, _ in results]
    hits = [recipe_id for recipe_id in returned if recipe_id in good]

    reciprocal_rank = 0.0
    for position, recipe_id in enumerate(returned, start=1):
        if recipe_id in good:
            reciprocal_rank = 1.0 / position
            break

    return {
        # None e non 0.0: sulle query negative una media deve rompersi, non sporcarsi
        "recall": len(expected & set(returned)) / len(expected) if expected else None,
        "precision": len(hits) / K,
        "precision_out": len(hits) / len(returned) if returned else 0.0,
        "rr": reciprocal_rank,
        "silent": len(returned) == 0,
        "returned": returned,
    }


demo = judge(queries[0], retrieve_overlap(queries[0]["pantry"]))
print(queries[0]["id"], {k: v for k, v in demo.items() if k != "returned"})

q1-classic {'recall': 1.0, 'precision': 0.6666666666666666, 'precision_out': 1.0, 'rr': 1.0, 'silent': False}


In [11]:
STRATEGIES = {
    "overlap-jaccard":  lambda p: retrieve_overlap(p, scorer="jaccard"),
    "overlap-coverage": lambda p: retrieve_overlap(p, scorer="coverage"),
    "overlap-count":    lambda p: retrieve_overlap(p, scorer="count"),
    "embed-A":          lambda p: retrieve_embed(p, variant="A"),
    "embed-A-key":      lambda p: retrieve_embed(p, variant="A-key"),
    "embed-B":          lambda p: retrieve_embed(p, variant="B"),
    "embed-C":          lambda p: retrieve_embed(p, variant="C"),
}

REPEATS = 5
results = {}

for name, strategy in STRATEGIES.items():
    per_query = {}
    for query in queries:
        strategy(query["pantry"])       # warm-up: la prima chiamata misurerebbe l'avvio
        times = []
        for _ in range(REPEATS):
            t0 = perf_counter()
            out = strategy(query["pantry"])
            times.append((perf_counter() - t0) * 1000)
        metrics = judge(query, out)
        metrics["ms"] = statistics.median(times)    # mediana: ignora la pausa del sistema
        metrics["scores"] = [round(score, 3) for _, score in out]
        per_query[query["id"]] = metrics
    results[name] = per_query

print("done:", ", ".join(results))

done: overlap-jaccard, overlap-coverage, overlap-count, embed-A, embed-A-key, embed-B, embed-C


In [12]:
NEGATIVE = [q["id"] for q in queries if not q["expected"]]
POSITIVE = [q["id"] for q in queries if q["expected"]]


def summarise(per_query):
    scored = [per_query[query_id] for query_id in POSITIVE]
    latencies = [m["ms"] for m in per_query.values()]
    return {
        "recall@3": statistics.mean(m["recall"] for m in scored),
        "prec@3": statistics.mean(m["precision"] for m in scored),
        "prec@out": statistics.mean(m["precision_out"] for m in scored),
        "MRR": statistics.mean(m["rr"] for m in scored),
        "silence": sum(per_query[query_id]["silent"] for query_id in NEGATIVE) / len(NEGATIVE),
        "ms_med": statistics.median(latencies),
        "ms_max": max(latencies),
    }


summary = {name: summarise(per_query) for name, per_query in results.items()}

head = f"{'strategy':18} {'recall@3':>9} {'prec@3':>7} {'prec@out':>9} {'MRR':>6} {'silence':>8} {'ms med':>7} {'ms max':>7}"
print(head)
print("-" * len(head))
for name, s in summary.items():
    print(f"{name:18} {s['recall@3']:>9.2f} {s['prec@3']:>7.2f} {s['prec@out']:>9.2f} "
          f"{s['MRR']:>6.2f} {s['silence']:>7.0%} {s['ms_med']:>7.2f} {s['ms_max']:>7.2f}")
print()
print(f"startup: model {load_ms:.0f} ms + encoding {encode_ms['A']:.0f} ms (embedding) vs 0 ms (overlap)")
print(f"recall over {len(POSITIVE)} queries, silence over {len(NEGATIVE)}, median latency over {REPEATS} runs")

strategy            recall@3  prec@3  prec@out    MRR  silence  ms med  ms max
------------------------------------------------------------------------------
overlap-jaccard         0.98    0.49      0.68   1.00    100%    0.01    0.01
overlap-coverage        1.00    0.49      0.68   1.00    100%    0.01    0.01
overlap-count           0.98    0.49      0.68   1.00    100%    0.00    0.01
embed-A                 0.91    0.47      0.47   0.87      0%   25.87   53.33
embed-A-key             0.96    0.47      0.47   0.97      0%   25.24   71.21
embed-B                 0.89    0.38      0.38   0.81      0%   22.89   34.66
embed-C                 0.96    0.47      0.47   1.00      0%   39.74   63.23

startup: model 17799 ms + encoding 1400 ms (embedding) vs 0 ms (overlap)
recall over 15 queries, silence over 2, median latency over 5 runs


In [13]:
MAIN = ["overlap-jaccard", "embed-A"]

for name in MAIN:
    print(f"\n=== {name} ===")
    for query in queries:
        m = results[name][query["id"]]
        expected = set(query["expected"])
        acceptable = set(query["acceptable"])
        shown = " ".join(
            f"{'*' if recipe_id in expected else '+' if recipe_id in acceptable else ' '}"
            f"#{recipe_id}({score})"
            for recipe_id, score in zip(m["returned"], m["scores"])
        ) or "(silent)"
        recall = "  -  " if m["recall"] is None else f"{m['recall']:.2f}"
        print(f"{query['id']:22} recall {recall}  rr {m['rr']:.2f}  {shown}")
print("\n* = expected, + = acceptable")


=== overlap-jaccard ===
q1-classic             recall 1.00  rr 1.00  *#3(1.0) +#18(0.5)
q2-tiramisu            recall 1.00  rr 1.00  *#1(0.5)
q3-strange-combination recall 1.00  rr 1.00  *#29(0.571)  #7(0.25)  #9(0.2)
q4-substitution        recall 1.00  rr 1.00  *#3(0.6)  #18(0.286)
q5-derivation          recall 1.00  rr 1.00  *#3(0.8) +#18(0.5)
q6-minimal             recall 1.00  rr 1.00  *#16(0.5)
q7-full-pantry         recall 0.67  rr 1.00  *#7(0.5) *#14(0.5)  #13(0.364)
q8-staple-only         recall   -    rr 0.00  (silent)
q9-negative            recall   -    rr 0.00  (silent)
q10-riso               recall 1.00  rr 1.00  *#8(0.444)
q11-two-risottos       recall 1.00  rr 1.00  *#12(0.714) +#17(0.556)  #2(0.182)
q12-soup               recall 1.00  rr 1.00  *#20(0.667) +#11(0.5) +#22(0.455)
q13-aubergine          recall 1.00  rr 1.00  *#13(0.625) +#14(0.429)  #2(0.3)
q14-chicken            recall 1.00  rr 1.00  *#15(0.667)  #22(0.333)  #2(0.273)
q15-baking             recall 1.00  r

In [14]:
print("Missed expected recipes\n")
for name in results:
    misses = []
    for query in queries:
        m = results[name][query["id"]]
        if m["recall"] is not None and m["recall"] < 1.0:
            lost = [r for r in query["expected"] if r not in m["returned"]]
            misses.append(f"   {query['id']:22} " + ", ".join(f"#{r} {titles[r]}" for r in lost))
    print(f"{name}: {len(misses)} incomplete")
    print("\n".join(misses) if misses else "   (none)")
    print()

print("\nScores on negative queries, against the range on the good ones")
for name in MAIN:
    for query_id in NEGATIVE:
        print(f"   {name:18} {query_id:16} {results[name][query_id]['scores'] or '(silent)'}")
positive_scores = [s for query_id in POSITIVE for s in results["embed-A"][query_id]["scores"]]
print(f"\n   embed-A on positive queries: min {min(positive_scores):.3f} "
      f"median {statistics.median(positive_scores):.3f} max {max(positive_scores):.3f}")

Missed expected recipes

overlap-jaccard: 1 incomplete
   q7-full-pantry         #25 Focaccia Genovese

overlap-coverage: 0 incomplete
   (none)

overlap-count: 1 incomplete
   q7-full-pantry         #25 Focaccia Genovese

embed-A: 2 incomplete
   q3-strange-combination #29 Pasta alle Vongole
   q7-full-pantry         #25 Focaccia Genovese

embed-A-key: 1 incomplete
   q7-full-pantry         #14 Pizza Margherita, #25 Focaccia Genovese

embed-B: 2 incomplete
   q5-derivation          #3 Pasta alla Carbonara
   q7-full-pantry         #14 Pizza Margherita, #25 Focaccia Genovese

embed-C: 1 incomplete
   q7-full-pantry         #14 Pizza Margherita, #25 Focaccia Genovese


Scores on negative queries, against the range on the good ones
   overlap-jaccard    q8-staple-only   (silent)
   overlap-jaccard    q9-negative      (silent)
   embed-A            q8-staple-only   [0.301, 0.267, 0.256]
   embed-A            q9-negative      [0.713, 0.599, 0.588]

   embed-A on positive queries: min 0.679

## Conclusions

The overlap wins, and not by a little. It finds practically the same recipes as the embedding
(0.98 against 0.96 recall) but answers in hundredths of a millisecond instead of tens of them,
and has nothing to load at startup, while the other one takes a good ten seconds. The exact
milliseconds change from run to run depending on what else is running on the machine, but the gap
stays at three orders of magnitude.

The reason is clear on `q3`. There are clams in the pantry, and the embedding does not bring up
Pasta alle Vongole: it puts Carbonara and Amatriciana first, which are Italian pasta dishes with a
similar ingredient list. The model looks at how much the pantry resembles the recipe overall, and
one ingredient more or less barely moves that resemblance. But having clams is not a matter of
resemblance, it is yes or no: either you have them or you are not making that dish. Counting the
ingredients in common answers exactly that question.

There is also the case where the right answer is not to answer. With `cacao` and `capperi` in the
pantry you cannot cook anything, and the overlap indeed stays silent. The embedding answers
instead, and it is not that it lacks the rule to keep quiet: on that query it gives 0.713, while
its correct answers go as low as 0.679. There is no threshold that cuts out the impossible queries
without throwing away good answers too.

What changes from now on: no vector database, and the Python worker does not need PyTorch nor to
download a model. I will bring embeddings back later for the app's memory, which is another
problem.

What this test does not say. It is 30 recipes, 80 possible ingredients, and the user picks them
from a list instead of typing them: it is the easiest case for the overlap and the hardest for
embeddings, which are useful above all when the text is free and messy. With 10.000 recipes or
with free text search the test would have to be redone.